# Counterfactuals — CelebA-HQ (Stable Diffusion 3)

Counterfactual editing on CelebA-HQ-simple with the seven-attribute SCM
(Smiling / Eyeglasses / Mouth_Slightly_Open / Male / Bald /
Wearing_Lipstick / Wearing_Hat) on top of a frozen Stable Diffusion 3
backbone. Two editing strategies are demonstrated: classic
Prompt-to-Prompt and the inverse-based FlowEdit-SD3.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
# Two extra entries on sys.path:
#   - the package root, so `causal_modules` / `causal_datasets` resolve.
#   - this notebook's own folder (kept symmetric with the SD1.5 notebooks).
sys.path.append(str(Path.cwd().resolve().parent))  # repo root
sys.path.append(str(Path.cwd().resolve()))  # this notebook's folder

import os
import random
from copy import deepcopy

import numpy as np
import torch
from PIL import Image
from torchvision import transforms

from diffusers.models.controlnets.controlnet_sd3_causal import Causal_SD3ControlNetModel
from diffusers.models.modeling_utils import load_state_dict
from diffusers.pipelines import StableDiffusion3InpaintPipeline_Adapter
from diffusers.utils.torch_utils import randn_tensor

from causal_datasets.celebahq_dataset import CelebAHQ
from causal_modules.ddim_modules_sd3 import (
    FlowEditSD3,
    encode_prompt_pai,
    load_mcpl_embeddings,
    load_tokenizers_text_encoders,
    save_images_grid,
    tokenize_prompt,
)


## 1. Configuration

All paths are derived from three roots:

* `MODEL_CACHE` — local HuggingFace snapshot cache holding the frozen
  Stable Diffusion 3 backbone.
* `LOGS_ROOT` — directory holding the SD3 training-run sub-folders
  produced by `train_SD3.py`.
* `DATA_ROOT` — root of the counterfactual-benchmark CelebA-HQ datasets.

Switching to a different cluster path or different training run only
requires editing this cell.


In [ ]:
# Shared roots
MODEL_CACHE = os.environ.get('MODEL_CACHE', '')
LOGS_ROOT = os.environ.get('LOGS_ROOT', '')
DATA_ROOT = os.environ.get('DATA_ROOT', '')

# 1) Frozen SD3-medium backbone.
BASE_MODEL_PATH = ''

# 2) Causal-Adapter SD3 ControlNet + matching MCPL learned pseudo-tokens.
CONTROLNET_PATH  = ''
EMBEDDING_1_PATH = ''
EMBEDDING_2_PATH = ''
EMBEDDING_3_PATH = ''

DATASET = 'celebahq_simple'
SIZE = 512
LOAD_DTYPE = torch.float16

PROMPT = 'a human of @ and * and mouth and gender and ** and $ and #'
PRESUDO_WORDS = '@,*,mouth,gender,**,$,#'
PRESUDO_LIST = PRESUDO_WORDS.split(',')

## 2. Build pipeline + transforms

Builds the shared image transforms, loads the SD3 Causal ControlNet,
attaches MCPL pseudo-token embeddings to the three text encoders
(CLIP-1, CLIP-2, T5), and assembles the
`StableDiffusion3InpaintPipeline_Adapter`.


In [ ]:
image_transforms = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
original_transforms = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BILINEAR),
])
tensor_image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

device = torch.device('cuda')

controlnet = Causal_SD3ControlNetModel.from_pretrained(CONTROLNET_PATH, torch_dtype=LOAD_DTYPE)
controlnet.eval()

# Load the three SD3 text encoders + tokenizers, then patch in the learned MCPL embeddings.
tokenizers, text_encoders = load_tokenizers_text_encoders(BASE_MODEL_PATH, LOAD_DTYPE)
text_encoders = load_mcpl_embeddings(
    [EMBEDDING_1_PATH, EMBEDDING_2_PATH, EMBEDDING_3_PATH],
    tokenizers, text_encoders, LOAD_DTYPE,
)

pipe = StableDiffusion3InpaintPipeline_Adapter.from_pretrained(
    BASE_MODEL_PATH, controlnet=controlnet,
    tokenizer=tokenizers[0], tokenizer_2=tokenizers[1], tokenizer_3=tokenizers[2],
    text_encoder=text_encoders[0], text_encoder_2=text_encoders[1], text_encoder_3=text_encoders[2],
    torch_dtype=LOAD_DTYPE,
)
pipe.safety_checker = None
pipe.requires_safety_checker = False
pipe = pipe.to(device)

presudo_token_ids_one   = tokenizers[0].encode(' '.join(PRESUDO_LIST), add_special_tokens=False)
presudo_token_ids_two   = tokenizers[1].encode(' '.join(PRESUDO_LIST), add_special_tokens=False)
presudo_token_ids_three = tokenizers[2].encode(' '.join(PRESUDO_LIST), add_special_tokens=False)

string_tokens_one   = tokenize_prompt(tokenizers[0], PROMPT, max_sequence_length=77)
string_tokens_two   = tokenize_prompt(tokenizers[1], PROMPT, max_sequence_length=77)
string_tokens_three = tokenize_prompt(tokenizers[2], PROMPT, max_sequence_length=77)


## 3. Pick a sample from the CelebA-HQ test set

Loads the seven binary CelebA-HQ-simple attributes (Smiling / Eyeglasses
/ Mouth_Slightly_Open / Male / Bald / Wearing_Lipstick / Wearing_Hat).
Edit `IMG_ID` to inspect a different image.


In [ ]:
def dataset_load_path(data_root, dataset, split='train'):
    data = CelebAHQ(root=data_root, split=split, transform=None, download=False)
    num_images = len(data)
    if 'simple' in dataset:
        selected_item = ['Smiling', 'Eyeglasses', 'Mouth_Slightly_Open',
                          'Male', 'Bald', 'Wearing_Lipstick', 'Wearing_Hat']
    else:
        raise AssertionError(f'no such {dataset} dataset')
    attribute_ids = [data.attr_names.index(attr) for attr in selected_item]
    metrics = {attr: torch.as_tensor(data.attr[:, attr_id], dtype=torch.float32)
               for attr, attr_id in zip(selected_item, attribute_ids)}
    imglabel = torch.cat([metrics[attr].unsqueeze(1) for attr in selected_item], dim=1)
    return data, imglabel, num_images


data, imglabel, num_images = dataset_load_path(data_root=DATA_ROOT, dataset=DATASET, split='test')

# paper image id (190, 8385, 2457, 2184, 3627, 5694)
# female (8385, 3393, 6942)
# valid (79, 145, 17, 18, 712)
# ablation (test2, test190, valid23, test25)
# limitation (test2, test357)
# pressure test (test8951, test1002)
IMG_ID = 8
img, label = data[IMG_ID][0], imglabel[IMG_ID].unsqueeze(0)
print(label)
img


## 4. Counterfactual editing — Prompt-to-Prompt

For each of the seven attributes, flip its value relative to the
training label and ask the SD3 pipeline to re-render the image. The
results are written to `intervention_variable{i}.png` next to this
notebook.


In [ ]:
# Prompt2Prompt
input_image = img
if input_image.mode != 'RGB':
    input_image = input_image.convert('RGB')

original_img = input_image.copy()
original_img = original_transforms(original_img)
image = image_transforms(input_image)

# do intervention
inter_value = 1 - label.clone().squeeze()

set_guidance_scale = 3.0
num_steps = 50
s_step = 0
invert_guidance_scale = 1.0
blend_word = True

image_lists = [np.asarray(original_img)]

shape = (1, 16, SIZE // 8, SIZE // 8)
generator = torch.Generator(0)
latents = randn_tensor(shape, generator=generator, device=pipe.device, dtype=LOAD_DTYPE)

with torch.no_grad():
    for inter_id in range(0, 7, 1):
        s_step = 0

        DSCM_labels = label.clone()
        DSCM_labels[:, inter_id] = inter_value[inter_id]
        # If we're intervening on attribute 0, enforce a dependency:
        # attr2 follows the (complemented) value of attr0, per sample.
        if inter_id == 0:
            DSCM_labels[:, 2] = inter_value[0]

        DSCM_labels = DSCM_labels.to(pipe.device)

        prompt_embeds, pooled_prompt_embeds = encode_prompt_pai(
            text_encoders=[pipe.text_encoder, pipe.text_encoder_2, pipe.text_encoder_3],
            tokenizers=[None, None, None],
            text_input_ids_list=[string_tokens_one, string_tokens_two, string_tokens_three],
            max_sequence_length=77,
            prompt=PROMPT,
            label=DSCM_labels.to(dtype=LOAD_DTYPE).unsqueeze(2),
            presudo_token_lists=[presudo_token_ids_one, presudo_token_ids_two, presudo_token_ids_three],
            weight_dtype=LOAD_DTYPE,
            is_negative_embeddings=False,
        )
        # null embeddings
        negative_prompt_embeds, negative_pooled_prompt_embeds = encode_prompt_pai(
            text_encoders=[pipe.text_encoder, pipe.text_encoder_2, pipe.text_encoder_3],
            tokenizers=[pipe.tokenizer, pipe.tokenizer_2, pipe.tokenizer_3],
            text_input_ids_list=[None, None, None],
            max_sequence_length=77,
            prompt='',
            label=DSCM_labels.to(dtype=LOAD_DTYPE).unsqueeze(2),
            presudo_token_lists=[presudo_token_ids_one, presudo_token_ids_two, presudo_token_ids_three],
            weight_dtype=LOAD_DTYPE,
            is_negative_embeddings=True,
        )

        if image.dim() == 3:
            image = image.unsqueeze(0)
        interved_image = pipe(
            image=image, height=SIZE, width=SIZE,
            generator=generator, latents=latents,
            guidance_scale=set_guidance_scale,
            num_inference_steps=num_steps,
            prompt_embeds=prompt_embeds,
            negative_prompt_embeds=negative_prompt_embeds,
            pooled_prompt_embeds=pooled_prompt_embeds,
            negative_pooled_prompt_embeds=negative_pooled_prompt_embeds,
        ).images[0]
        image_lists.append(np.asarray(interved_image))

    output = './'
    save_path = os.path.join(output, f'intervention_variable{inter_id}.png')
    save_images_grid([image_lists], (1, 8), save_path)
    print(f'save imgs in {save_path}')


## 5. Counterfactual editing — Inverse-based (FlowEdit-SD3)

Same intervention sweep, but using inverse-based FlowEdit-SD3 instead of
classic Prompt-to-Prompt. Re-runs the pre-trained adapter against the
chosen image (override `IMG_ID` above to inspect a different sample).


In [ ]:
from copy import deepcopy

T_steps = 50
n_avg = 1
src_guidance_scale = 1.5
tar_guidance_scale = 2.5
n_min = 10
n_max = 50
seed = 49

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = pipe.device
inter_value = 1 - label.clone().squeeze()

input_image = img
if input_image.mode != 'RGB':
    input_image = input_image.convert('RGB')

original_img = input_image.copy()
original_img = original_transforms(original_img)
image = image_transforms(input_image)

image_src = pipe.image_processor.preprocess(image)
image_src = image_src.to(device).to(dtype=LOAD_DTYPE)

image_lists = [np.asarray(original_img)]
with torch.autocast('cuda'), torch.inference_mode():
    x0_src_denorm = pipe.vae.encode(image_src).latent_dist.mode()
x0_src = (x0_src_denorm - pipe.vae.config.shift_factor) * pipe.vae.config.scaling_factor
x0_src = x0_src.to(device)

# Subset of attributes the FlowEdit demo iterates over.
for inter_id in [0, 1, 2]:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if inter_id in [6]:
        src_guidance_scale = 1.0
        tar_guidance_scale = 2.5
        n_min = 10
        n_max = 50
    elif inter_id in [3]:
        src_guidance_scale = 1.0
        tar_guidance_scale = 1.5
        n_min = 10
        n_max = 50
    else:
        src_guidance_scale = 1.0
        tar_guidance_scale = 1.8
        n_min = 0
        n_max = 50

    DSCM_labels = label.clone()
    DSCM_labels[:, inter_id] = inter_value[inter_id]
    if inter_id == 0:
        DSCM_labels[:, 2] = inter_value[0]
    DSCM_labels = DSCM_labels.to(pipe.device)

    x0_tar = FlowEditSD3(
        pipe, pipe.scheduler, x0_src.clone(),
        PROMPT, PRESUDO_LIST,
        label.clone().to(pipe.device),
        DSCM_labels.to(pipe.device),
        negative_prompt='',
        T_steps=T_steps, n_avg=n_avg,
        src_guidance_scale=src_guidance_scale,
        tar_guidance_scale=tar_guidance_scale,
        n_min=n_min, n_max=n_max,
    )

    x0_tar_denorm = (x0_tar / pipe.vae.config.scaling_factor) + pipe.vae.config.shift_factor
    with torch.autocast('cuda'), torch.inference_mode():
        image_tar = pipe.vae.decode(x0_tar_denorm, return_dict=False)[0]
        image_tar = pipe.image_processor.postprocess(image_tar)

    image_lists.append(np.asarray(image_tar[0]))

output = './'
save_path = os.path.join(output, f'intervention_variable{inter_id}.png')
save_images_grid([image_lists], (1, 8), save_path)
print(f'save imgs in {save_path}')
